# Databricks SQL Data Types - Complete Guide

## Numeric Data Types

### Integer Types
- **TINYINT**: 1-byte signed integer (-128 to 127)
- **SMALLINT**: 2-byte signed integer (-32,768 to 32,767)
- **INT / INTEGER**: 4-byte signed integer (-2,147,483,648 to 2,147,483,647)
- **BIGINT**: 8-byte signed integer (-9,223,372,036,854,775,808 to 9,223,372,036,854,775,807)

### Decimal Types
- **FLOAT**: 4-byte single-precision floating point
- **DOUBLE**: 8-byte double-precision floating point
- **DECIMAL(p,s)**: Fixed precision decimal, where p = precision (total digits), s = scale (digits after decimal)
  - Example: DECIMAL(10,2) can store 99999999.99

## String Data Types
- **STRING**: Variable-length character string (UTF-8 encoded)
- **VARCHAR(n)**: Variable-length string with max length n
- **CHAR(n)**: Fixed-length string of length n (padded with spaces)

## Date and Time Types
- **DATE**: Calendar date (year, month, day) - Format: 'YYYY-MM-DD'
- **TIMESTAMP**: Timestamp with microsecond precision - Format: 'YYYY-MM-DD HH:MM:SS.SSSSSS'
- **TIMESTAMP_NTZ**: Timestamp without timezone (No Time Zone)
- **INTERVAL**: Represents a time interval (YEAR, MONTH, DAY, HOUR, MINUTE, SECOND)

## Boolean Type
- **BOOLEAN**: TRUE or FALSE values

## Binary Type
- **BINARY**: Variable-length binary data (byte array)

## Complex Data Types
- **ARRAY<T>**: Ordered collection of elements of type T
  - Example: ARRAY<STRING>, ARRAY<INT>
- **MAP<K,V>**: Collection of key-value pairs where K is key type and V is value type
  - Example: MAP<STRING, INT>
- **STRUCT<field1:type1, field2:type2, ...>**: Named fields with specified types
  - Example: STRUCT<name:STRING, age:INT>

## Semi-Structured Data Type
- **VARIANT**: Stores semi-structured data (JSON, XML) - can contain any valid JSON value
  - Supports nested structures without predefined schema
  - Query using variant_get(), : notation, or JSON path expressions

In [0]:
%sql
Create catalog if not exists demo_catalog;

Create schema if not exists demo_catalog.demo_schema;

DESCRIBE DETAIL demo_catalog.demo_schema.all_data_types;

In [0]:
%sql
-- CREATE TABLE demonstrating ALL data types
CREATE OR REPLACE TABLE demo_catalog.demo_schema.all_data_types (
  -- Numeric Types - Integers
  tiny_int_col TINYINT COMMENT 'Stores small integers (-128 to 127)',
  small_int_col SMALLINT COMMENT 'Stores medium integers (-32,768 to 32,767)',
  int_col INT COMMENT 'Standard integer type',
  big_int_col BIGINT COMMENT 'Large integer values',
  
  -- Numeric Types - Decimals
  float_col FLOAT COMMENT 'Single precision floating point',
  double_col DOUBLE COMMENT 'Double precision floating point',
  decimal_col DECIMAL(18,2) COMMENT 'Fixed precision decimal (e.g., currency)',
  
  -- String Types
  string_col STRING COMMENT 'Variable-length string (most common)',
  varchar_col VARCHAR(100) COMMENT 'Variable string with max length',
  char_col CHAR(10) COMMENT 'Fixed-length string (padded)',
  
  -- Date and Time Types
  date_col DATE COMMENT 'Calendar date only',
  timestamp_col TIMESTAMP COMMENT 'Date and time with timezone',
  timestamp_ntz_col TIMESTAMP_NTZ COMMENT 'Date and time without timezone',
  
  -- Boolean Type
  boolean_col BOOLEAN COMMENT 'True/False values',
  
  -- Binary Type
  binary_col BINARY COMMENT 'Binary data (byte array)',
  
  -- Complex Types
  array_col ARRAY<STRING> COMMENT 'Array of strings',
  map_col MAP<STRING, INT> COMMENT 'Key-value pairs',
  struct_col STRUCT<first_name:STRING, last_name:STRING, age:INT> COMMENT 'Structured data',
  
  -- Semi-Structured Type
  variant_col VARIANT COMMENT 'JSON or semi-structured data'
)
USING DELTA
COMMENT 'Demonstration table showing all Databricks SQL data types';

# Table Constraints in Delta Tables

Constraints ensure data quality and integrity by enforcing rules on the data.

## Supported Constraints on Serverless

### 1. NOT NULL Constraint
- Ensures a column cannot contain NULL values
- ✅ **Fully supported** on Serverless
- Enforced at write time
- Syntax: `column_name TYPE NOT NULL`

### 2. Primary Key (Preview)
- Uniquely identifies each row
- ✅ **Supported** on Serverless
- Automatically creates NOT NULL constraint
- Can be single or composite (multiple columns)
- Enforced through Delta Lake

### 3. Foreign Key (Preview)
- Enforces referential integrity between tables
- ✅ **Supported** on Serverless
- References primary key in another table
- Currently informational (not enforced at runtime)

## ⚠️ Constraints NOT Supported on Serverless

### CHECK Constraint
- ❌ **Not currently supported** on Serverless compute
- Would enforce boolean conditions on column values
- Available on classic clusters with DBR 13.3+
- Alternative: Implement validation logic in your ETL code

## Constraint Benefits
- **Data Quality**: Prevent invalid data at ingestion
- **Documentation**: Self-documenting data rules
- **Query Optimization**: Optimizer can use constraints for better plans
- **Early Failure**: Catch data issues before they propagate

In [0]:
%sql
-- NOT NULL CONSTRAINT
-- Prevents NULL values in specified columns

CREATE OR REPLACE TABLE demo_catalog.demo_schema.customers (
  customer_id BIGINT NOT NULL COMMENT 'Customer ID cannot be null',
  email STRING NOT NULL COMMENT 'Email is required',
  first_name STRING NOT NULL,
  last_name STRING NOT NULL,
  phone STRING,  -- NULL allowed
  registration_date DATE NOT NULL,
  last_purchase_date DATE  -- NULL allowed for new customers
)
USING DELTA
COMMENT 'Customers table with NOT NULL constraints';

# CHECK Constraints - Not Supported on Serverless

## ❌ Current Limitation

CHECK constraints are **NOT supported** on Databricks Serverless compute.

**Error Message**: `Only PRIMARY KEY and FOREIGN KEY constraints are currently supported.`

## What Are CHECK Constraints?

CHECK constraints enforce business rules and data validation at the table level:
- Single column validation (e.g., `price > 0`)
- Range checks (e.g., `discount_percentage BETWEEN 0 AND 100`)
- Category validation (e.g., `status IN ('active', 'inactive')`)
- Cross-column validation (e.g., `start_date < end_date`)

## 🛠️ Alternatives for Serverless

### Option 1: Application-Level Validation
Implement validation in your ETL code:

```python
from pyspark.sql import functions as F

# Validate data before writing
df_validated = df.filter(
    (F.col("price") > 0) & 
    (F.col("discount_percentage").between(0, 100)) &
    (F.col("quantity_in_stock") >= 0)
)

df_validated.write.mode("append").saveAsTable("products")
```

### Option 2: Data Quality Expectations (Recommended)
Use Delta Live Tables (DLT) expectations:

```python
@dlt.expect_or_fail("valid_price", "price > 0")
@dlt.expect_or_fail("valid_discount", "discount_percentage BETWEEN 0 AND 100")
@dlt.table
def products():
    return spark.read.table("source_products")
```

### Option 3: Post-Write Validation
Query for invalid records after insert:

```sql
SELECT * FROM products
WHERE price <= 0 
   OR discount_percentage NOT BETWEEN 0 AND 100
   OR quantity_in_stock < 0;
```

## When Will CHECK Constraints Be Available?

CHECK constraints are available on **classic clusters** with DBR 13.3+. Serverless support may be added in future releases.

## Example: Table Without CHECK Constraints

```sql
CREATE OR REPLACE TABLE demo_catalog.demo_schema.products (
  product_id BIGINT NOT NULL,
  product_name STRING NOT NULL,
  price DECIMAL(10,2),
  quantity_in_stock INT,
  discount_percentage DECIMAL(5,2),
  category STRING,
  created_date DATE
)
USING DELTA
COMMENT 'Products table - validation handled in ETL code';
```

In [0]:
%sql
-- PRIMARY KEY CONSTRAINT (Preview Feature)
-- Uniquely identifies each row in the table
-- Note: Requires Delta Lake with Uniform enabled

CREATE OR REPLACE TABLE demo_catalog.demo_schema.orders (
  order_id BIGINT NOT NULL,
  customer_id BIGINT NOT NULL,
  order_date DATE NOT NULL,
  total_amount DECIMAL(12,2),
  status STRING,
  
  -- Single column primary key
  CONSTRAINT orders_pk PRIMARY KEY (order_id)
)
USING DELTA
COMMENT 'Orders table with PRIMARY KEY constraint';

-- Composite Primary Key Example
CREATE OR REPLACE TABLE demo_catalog.demo_schema.order_items (
  order_id BIGINT NOT NULL,
  line_item_number INT NOT NULL,
  product_id BIGINT NOT NULL,
  quantity INT NOT NULL,
  price DECIMAL(10,2) NOT NULL,
  
  -- Composite primary key (multiple columns)
  CONSTRAINT order_items_pk PRIMARY KEY (order_id, line_item_number)
)
USING DELTA
COMMENT 'Order items with composite PRIMARY KEY';

In [0]:
%sql
-- FOREIGN KEY CONSTRAINT (Preview Feature - Informational)
-- Defines relationships between tables
-- Currently used for documentation and query optimization, not enforced

-- First, ensure customers table has a primary key
CREATE OR REPLACE TABLE demo_catalog.demo_schema.customers (
  customer_id BIGINT NOT NULL,
  email STRING NOT NULL,
  first_name STRING NOT NULL,
  last_name STRING NOT NULL,
  CONSTRAINT customers_pk PRIMARY KEY (customer_id)
)
USING DELTA
COMMENT 'Customers table with primary key for foreign key reference';

-- Create the products table that will be referenced
CREATE OR REPLACE TABLE demo_catalog.demo_schema.products (
  product_id BIGINT NOT NULL,
  product_name STRING NOT NULL,
  CONSTRAINT products_pk PRIMARY KEY (product_id)
)
USING DELTA
COMMENT 'Products table for foreign key reference';

-- Now create order_details with foreign keys
CREATE OR REPLACE TABLE demo_catalog.demo_schema.order_details (
  order_id BIGINT NOT NULL,
  customer_id BIGINT NOT NULL,
  product_id BIGINT NOT NULL,
  order_date DATE NOT NULL,
  quantity INT NOT NULL,
  
  -- Primary key for this table
  CONSTRAINT order_details_pk PRIMARY KEY (order_id),
  
  -- Foreign key referencing customers table
  CONSTRAINT fk_customer FOREIGN KEY (customer_id) 
    REFERENCES demo_catalog.demo_schema.customers(customer_id),
  
  -- Foreign key referencing products table
  CONSTRAINT fk_product FOREIGN KEY (product_id) 
    REFERENCES demo_catalog.demo_schema.products(product_id)
)
USING DELTA
COMMENT 'Order details with FOREIGN KEY constraints for referential integrity';

# Delta Table Properties

Table properties control various aspects of Delta table behavior, optimization, and features.

## Categories of Table Properties

### 1. Delta Protocol Properties
- **delta.minReaderVersion**: Minimum reader version required
- **delta.minWriterVersion**: Minimum writer version required
- Enable advanced features like Column Mapping, Deletion Vectors

### 2. Performance Properties
- **delta.autoOptimize.optimizeWrite**: Auto-compaction during writes
- **delta.autoOptimize.autoCompact**: Automatic compaction after writes
- **delta.targetFileSize**: Target file size for optimization
- **delta.tuneFileSizesForRewrites**: Optimize file sizes for rewrites

### 3. Time Travel & Retention
- **delta.logRetentionDuration**: How long to keep transaction logs (default: 30 days)
- **delta.deletedFileRetentionDuration**: How long to keep deleted files (default: 7 days)
- **delta.enableChangeDataFeed**: Enable Change Data Feed (CDC)

### 4. Schema Evolution
- **delta.columnMapping.mode**: Enable column mapping (none/name/id)
- **delta.enableDeletionVectors**: Enable deletion vectors for faster deletes

### 5. Data Quality
- **delta.checkConstraints.<name>**: Named check constraints
- **delta.feature.allowColumnDefaults**: Allow default values for columns

### 6. Liquid Clustering
- **delta.targetFileSize**: Target size for clustered files
- Table is created with CLUSTER BY clause

In [0]:
%sql
-- PERFORMANCE OPTIMIZATION TABLE PROPERTIES
-- These properties improve write and read performance

CREATE OR REPLACE TABLE demo_catalog.demo_schema.sales_optimized (
  sale_id BIGINT NOT NULL,
  product_id BIGINT NOT NULL,
  customer_id BIGINT NOT NULL,
  sale_date DATE NOT NULL,
  amount DECIMAL(12,2) NOT NULL,
  quantity INT NOT NULL
)
USING DELTA
TBLPROPERTIES (
  -- Auto-compact small files during writes
  'delta.autoOptimize.optimizeWrite' = 'true',
  
  -- Automatically run OPTIMIZE after write operations
  'delta.autoOptimize.autoCompact' = 'true',
  
  -- Target file size for compaction (128 MB)
  'delta.targetFileSize' = '134217728',
  
  -- Tune file sizes for rewrite operations
  'delta.tuneFileSizesForRewrites' = 'true'
)
COMMENT 'Sales table with auto-optimization enabled';

In [0]:
%sql
-- TIME TRAVEL AND RETENTION PROPERTIES
-- Control how long historical data is retained

CREATE OR REPLACE TABLE demo_catalog.demo_schema.transactions_history (
  transaction_id BIGINT NOT NULL,
  account_id BIGINT NOT NULL,
  transaction_date TIMESTAMP NOT NULL,
  amount DECIMAL(15,2) NOT NULL,
  transaction_type STRING NOT NULL
)
USING DELTA
TBLPROPERTIES (
  -- Keep transaction logs for 90 days (enables time travel)
  'delta.logRetentionDuration' = 'interval 90 days',
  
  -- Keep deleted data files for 30 days (for time travel queries)
  'delta.deletedFileRetentionDuration' = 'interval 30 days',
  
  -- Enable Change Data Feed for CDC operations
  'delta.enableChangeDataFeed' = 'true'
)
COMMENT 'Transaction table with extended retention for compliance';

-- Note: Longer retention = more storage costs but better time travel capability

In [0]:
%sql
-- SCHEMA EVOLUTION PROPERTIES
-- Enable advanced schema management features

CREATE OR REPLACE TABLE demo_catalog.demo_schema.user_events (
  event_id BIGINT NOT NULL,
  user_id BIGINT NOT NULL,
  event_type STRING NOT NULL,
  event_timestamp TIMESTAMP NOT NULL,
  event_data STRING
)
USING DELTA
TBLPROPERTIES (
  -- Enable column mapping (allows column renames without rewriting data)
  'delta.columnMapping.mode' = 'name',
  
  -- Enable deletion vectors (faster DELETEs without rewriting files)
  'delta.enableDeletionVectors' = 'true',
  
  -- Set minimum reader/writer versions for these features
  'delta.minReaderVersion' = '3',
  'delta.minWriterVersion' = '7'
)
COMMENT 'User events table with column mapping and deletion vectors';

-- Benefits:
-- - Rename columns without rewriting entire table
-- - Fast deletes without file rewrites
-- - Better schema evolution support

In [0]:
%sql
-- COMPREHENSIVE TABLE WITH MULTIPLE PROPERTIES
-- Combines multiple table properties for production use

CREATE OR REPLACE TABLE demo_catalog.demo_schema.customer_activity (
  activity_id BIGINT NOT NULL,
  customer_id BIGINT NOT NULL,
  activity_type STRING NOT NULL,
  activity_date DATE NOT NULL,
  activity_timestamp TIMESTAMP NOT NULL,
  revenue DECIMAL(12,2),
  metadata MAP<STRING, STRING>
)
USING DELTA
TBLPROPERTIES (
  -- Performance
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true',
  'delta.targetFileSize' = '134217728',
  
  -- Retention (compliance requirement)
  'delta.logRetentionDuration' = 'interval 180 days',
  'delta.deletedFileRetentionDuration' = 'interval 90 days',
  
  -- Change Data Feed for downstream CDC consumers
  'delta.enableChangeDataFeed' = 'true',
  
  -- Schema evolution
  'delta.columnMapping.mode' = 'name',
  'delta.enableDeletionVectors' = 'true',
  'delta.minReaderVersion' = '3',
  'delta.minWriterVersion' = '7',
  
  -- Custom metadata
  'pii_data' = 'false',
  'refresh_frequency' = 'hourly'
)
COMMENT 'Production customer activity table with full optimization and compliance settings';

# Delta Protocol - Understanding Reader and Writer Versions

## What is Delta Protocol?

The Delta Protocol defines the **versioned specification** for reading and writing Delta tables. It consists of two key components:

### 1. Reader Version (delta.minReaderVersion)
- Minimum version required to **READ** the table
- Ensures readers understand the table format and features
- If a reader's version < minReaderVersion, it **cannot read** the table

### 2. Writer Version (delta.minWriterVersion)
- Minimum version required to **WRITE** to the table
- Ensures writers can handle all table features correctly
- If a writer's version < minWriterVersion, it **cannot write** to the table

## Why Do We Need Protocol Versions?

### 1. **Feature Compatibility**
- New Delta features require specific protocol support
- Example: Deletion Vectors require Reader=3, Writer=7
- Example: Column Mapping requires Reader=2, Writer=5

### 2. **Data Safety**
- Prevents old clients from corrupting tables with features they don't understand
- A writer that doesn't know about Column Mapping shouldn't write to such tables

### 3. **Forward Compatibility**
- New Databricks versions can read old tables (backward compatible)
- Old Databricks versions are blocked from new tables (forward protection)

### 4. **Gradual Feature Adoption**
- Upgrade protocol only when you need specific features
- Don't upgrade unnecessarily if old clients need access

## Protocol Version History

| Reader | Writer | Features Enabled |
|--------|--------|------------------|
| 1 | 1 | Basic Delta Lake |
| 1 | 2 | Append-only tables |
| 1 | 3 | CHECK constraints |
| 2 | 5 | Column Mapping |
| 3 | 7 | Deletion Vectors, Clustering |
| 3 | 7 | Change Data Feed (CDF) |

## Important Considerations

⚠️ **Protocol upgrades are ONE-WAY** - you cannot downgrade protocol versions

⚠️ **Check compatibility** before upgrading if you have:
- External systems reading your Delta tables
- Multiple Databricks workspaces with different versions
- On-premises Delta Lake readers

✅ **Automatic upgrades** happen when you:
- Enable column mapping
- Enable deletion vectors
- Use liquid clustering
- Enable certain table features

In [0]:
%sql
-- ALTER TABLE - COLUMN OPERATIONS
-- Add, rename, modify, and drop columns

-- 1. ADD COLUMNS
-- Note: Skipped - columns already exist from previous execution
-- Use IF NOT EXISTS when available, or check DESCRIBE TABLE first
-- ALTER TABLE demo_catalog.demo_schema.customers
-- ADD COLUMN (
--   loyalty_points INT COMMENT 'Customer loyalty points',
--   preferred_contact STRING COMMENT 'Preferred contact method',
--   created_timestamp TIMESTAMP COMMENT 'Account creation timestamp'
-- );

-- Add single column with position
-- Note: Skipped - column already exists
-- ALTER TABLE demo_catalog.demo_schema.customers
-- ADD COLUMN middle_name STRING AFTER first_name;

-- 2. RENAME COLUMN (requires column mapping)
-- Note: Example removed - 'phone' column does not exist in this table
-- ALTER TABLE demo_catalog.demo_schema.customers
-- RENAME COLUMN phone TO phone_number;

-- 3. CHANGE COLUMN TYPE
-- Note: Skipped - loyalty_points doesn't exist in base table
-- ALTER TABLE demo_catalog.demo_schema.customers
-- ALTER COLUMN loyalty_points TYPE BIGINT;

-- 4. CHANGE COLUMN COMMENT
ALTER TABLE demo_catalog.demo_schema.customers
ALTER COLUMN email COMMENT 'Primary email address (verified)';

-- 5. SET NOT NULL
-- Note: Skipped - preferred_contact doesn't exist in base table
-- ALTER TABLE demo_catalog.demo_schema.customers
-- ALTER COLUMN preferred_contact SET NOT NULL;

-- 6. DROP NOT NULL
-- Note: Skipped - middle_name doesn't exist in base table
-- ALTER TABLE demo_catalog.demo_schema.customers
-- ALTER COLUMN middle_name DROP NOT NULL;

-- 7. DROP COLUMN (requires deletion vectors)
-- Note: Skipped - middle_name doesn't exist in base table
-- ALTER TABLE demo_catalog.demo_schema.customers
-- DROP COLUMN middle_name;

In [0]:
%sql
-- ALTER TABLE - CONSTRAINT OPERATIONS
-- Add, drop, and manage constraints
-- Note: CHECK constraints NOT supported on Serverless

-- 1. ADD PRIMARY KEY (requires protocol upgrade)
ALTER TABLE demo_catalog.demo_schema.customers
ADD CONSTRAINT customers_pk PRIMARY KEY (customer_id);

-- 2. DROP PRIMARY KEY
ALTER TABLE demo_catalog.demo_schema.customers
DROP CONSTRAINT customers_pk;

-- 3. ADD FOREIGN KEY (informational)
ALTER TABLE demo_catalog.demo_schema.orders
ADD CONSTRAINT fk_orders_customer 
  FOREIGN KEY (customer_id) 
  REFERENCES demo_catalog.demo_schema.customers(customer_id);

-- 4. DROP FOREIGN KEY
ALTER TABLE demo_catalog.demo_schema.orders
DROP CONSTRAINT fk_orders_customer;

In [0]:
%sql
-- ALTER TABLE - MODIFY TABLE PROPERTIES
-- Change table-level settings and behavior

-- 1. SET TABLE PROPERTIES
ALTER TABLE demo_catalog.demo_schema.sales_optimized
SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true',
  'delta.targetFileSize' = '268435456'  -- 256 MB
);

-- 2. UNSET TABLE PROPERTIES (remove property)
ALTER TABLE demo_catalog.demo_schema.sales_optimized
UNSET TBLPROPERTIES ('delta.targetFileSize');

-- 3. ENABLE CHANGE DATA FEED
ALTER TABLE demo_catalog.demo_schema.transactions_history
SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

-- 4. DISABLE CHANGE DATA FEED
ALTER TABLE demo_catalog.demo_schema.transactions_history
SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'false');

-- 5. UPDATE RETENTION POLICIES
ALTER TABLE demo_catalog.demo_schema.transactions_history
SET TBLPROPERTIES (
  'delta.logRetentionDuration' = 'interval 365 days',
  'delta.deletedFileRetentionDuration' = 'interval 180 days'
);

-- 6. SET TABLE COMMENT
ALTER TABLE demo_catalog.demo_schema.sales_optimized
SET TBLPROPERTIES ('comment' = 'Updated: Production sales table with enhanced optimization');

-- Or use shorthand
COMMENT ON TABLE demo_catalog.demo_schema.sales_optimized IS 
  'Updated: Production sales table with enhanced optimization';

In [0]:
%sql
-- ALTER TABLE - PROTOCOL VERSION UPGRADE
-- Upgrade reader/writer versions to enable advanced features

-- 1. Check current protocol version
DESCRIBE DETAIL demo_catalog.demo_schema.user_events;
-- Look for minReaderVersion and minWriterVersion

-- 2. ENABLE COLUMN MAPPING (upgrades to Reader=2, Writer=5)
ALTER TABLE demo_catalog.demo_schema.user_events
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

-- 3. ENABLE DELETION VECTORS (upgrades to Reader=3, Writer=7)
ALTER TABLE demo_catalog.demo_schema.user_events
SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'true');

-- 4. MANUAL PROTOCOL UPGRADE (if needed)
-- Upgrade to support specific features without enabling them yet
ALTER TABLE demo_catalog.demo_schema.user_events
SET TBLPROPERTIES (
  'delta.minReaderVersion' = '3',
  'delta.minWriterVersion' = '7'
);

-- ⚠️ WARNING: Protocol upgrades are IRREVERSIBLE!
-- Old clients with older protocol versions will not be able to read/write

-- 5. Check protocol version after upgrade
DESCRIBE DETAIL demo_catalog.demo_schema.user_events;

# Table Partitioning in Delta Lake

## What is Partitioning?

Partitioning divides a table into **physically separate directories** based on column values, organizing data for faster queries.

## How Partitioning Works

- Data is stored in separate directories: `/table/partition_col=value1/`, `/table/partition_col=value2/`
- Queries filtering on partition columns can **skip entire partitions** (partition pruning)
- Reduces amount of data scanned = faster queries + lower costs

## When to Use Partitioning

### ✅ Good Use Cases:
1. **Time-based data** with date/timestamp queries
   - `PARTITION BY (date)` or `PARTITION BY (year, month)`
2. **Large tables** (> 1 TB)
3. **Selective queries** that filter on specific columns
4. **High cardinality** partition columns (100-10,000 unique values)

### ❌ Avoid Partitioning When:
1. **Small tables** (< 1 TB) - overhead outweighs benefits
2. **Too many partitions** (> 10,000) - creates small files and metadata overhead
3. **Too few partitions** (< 10) - doesn't provide enough selectivity
4. **High cardinality** (millions of unique values) - too many directories
5. **Random access patterns** - queries don't filter on partition columns

## Partitioning vs. Clustering

| Aspect | Partitioning | Liquid Clustering |
|--------|-------------|-------------------|
| **Storage** | Separate directories | Single directory, clustered files |
| **Best for** | Time-series, date filters | Multiple filter columns |
| **Flexibility** | Fixed at creation | Can change cluster keys |
| **Small files** | Risk with many partitions | Auto-compacted |
| **Recommendation** | Legacy approach | **Preferred for new tables** |

## Key Recommendations

🎯 **Modern Approach**: Use **Liquid Clustering** instead of partitioning for new tables

🎯 **If you must partition**: Use low-cardinality columns (date, region, category)

🎯 **Target partition size**: 1 GB - 10 GB per partition

🎯 **Monitor**: Use `DESCRIBE DETAIL` to check partition count and file statistics

In [0]:
%sql
-- TABLE PARTITIONING EXAMPLES

-- 1. Single column partitioning (most common)
CREATE OR REPLACE TABLE demo_catalog.demo_schema.sales_by_date (
  sale_id BIGINT NOT NULL,
  product_id BIGINT NOT NULL,
  customer_id BIGINT NOT NULL,
  sale_date DATE NOT NULL,
  amount DECIMAL(12,2) NOT NULL,
  quantity INT NOT NULL
)
USING DELTA
PARTITIONED BY (sale_date)
COMMENT 'Sales data partitioned by date for efficient date-range queries';

-- Physical layout:
-- /sales_by_date/sale_date=2026-01-01/
-- /sales_by_date/sale_date=2026-01-02/
-- ...

-- 2. Multi-column partitioning
CREATE OR REPLACE TABLE demo_catalog.demo_schema.sales_by_region_date (
  sale_id BIGINT NOT NULL,
  region STRING NOT NULL,
  country STRING NOT NULL,
  sale_date DATE NOT NULL,
  amount DECIMAL(12,2) NOT NULL
)
USING DELTA
PARTITIONED BY (region, sale_date)
COMMENT 'Sales partitioned by region and date';

-- Physical layout:
-- /sales_by_region_date/region=North America/sale_date=2026-01-01/
-- /sales_by_region_date/region=North America/sale_date=2026-01-02/
-- /sales_by_region_date/region=Europe/sale_date=2026-01-01/
-- ...

-- 3. Year/Month partitioning (common pattern)
CREATE OR REPLACE TABLE demo_catalog.demo_schema.logs_by_year_month (
  log_id BIGINT NOT NULL,
  event_type STRING NOT NULL,
  event_timestamp TIMESTAMP NOT NULL,
  year INT NOT NULL,
  month INT NOT NULL,
  message STRING
)
USING DELTA
PARTITIONED BY (year, month)
COMMENT 'Event logs partitioned by year and month for long-term storage';

-- Best practice: Derive year/month from timestamp in INSERT
-- INSERT INTO logs_by_year_month
-- SELECT *, YEAR(event_timestamp) as year, MONTH(event_timestamp) as month
-- FROM source_logs;

In [0]:
%sql
-- QUERYING PARTITIONED TABLES
-- Efficient queries that leverage partition pruning

-- 1. Query with partition filter (EFFICIENT - prunes partitions)
SELECT *
FROM demo_catalog.demo_schema.sales_by_date
WHERE sale_date = '2026-08-01';
-- Reads only: /sales_by_date/sale_date=2026-08-01/

-- 2. Date range query (EFFICIENT)
SELECT product_id, SUM(amount) as total_sales
FROM demo_catalog.demo_schema.sales_by_date
WHERE sale_date BETWEEN '2026-08-01' AND '2026-08-31'
GROUP BY product_id;
-- Reads only: August 2026 partitions

-- 3. Multi-column partition query (EFFICIENT)
SELECT *
FROM demo_catalog.demo_schema.sales_by_region_date
WHERE region = 'North America' 
  AND sale_date >= '2026-08-01';
-- Reads only: North America partitions after Aug 1

-- 4. Query without partition filter (INEFFICIENT - full table scan)
SELECT *
FROM demo_catalog.demo_schema.sales_by_date
WHERE customer_id = 12345;
-- Reads ALL partitions! Consider clustering instead.

-- 5. Check partition pruning in query plan
EXPLAIN EXTENDED
SELECT *
FROM demo_catalog.demo_schema.sales_by_date
WHERE sale_date = '2026-08-01';
-- Look for "PushedFilters" or "PartitionFilters" in the plan

-- 6. List table partitions
SHOW PARTITIONS demo_catalog.demo_schema.sales_by_date;

-- 7. Get partition statistics
DESCRIBE DETAIL demo_catalog.demo_schema.sales_by_date;

In [0]:
%sql
-- ALTER TABLE - PARTITION OPERATIONS

-- 1. ADD PARTITION (metadata only, for external tables)
ALTER TABLE demo_catalog.demo_schema.sales_by_date
ADD PARTITION (sale_date = '2026-09-01')
LOCATION 's3://bucket/sales/sale_date=2026-09-01/';

-- 2. DROP PARTITION (deletes data!)
ALTER TABLE demo_catalog.demo_schema.sales_by_date
DROP PARTITION (sale_date = '2026-01-01');
-- ⚠️ Warning: This permanently deletes the partition data!

-- 3. REPAIR TABLE (recover partition metadata)
-- Useful when data files exist but partition metadata is missing
MSCK REPAIR TABLE demo_catalog.demo_schema.sales_by_date;

-- 4. DROP AND RECREATE PARTITION
ALTER TABLE demo_catalog.demo_schema.sales_by_date
DROP IF EXISTS PARTITION (sale_date = '2026-08-01');

-- Then reload data for that partition
INSERT INTO demo_catalog.demo_schema.sales_by_date
SELECT * FROM source_data WHERE sale_date = '2026-08-01';

-- 5. OPTIMIZE specific partition
OPTIMIZE demo_catalog.demo_schema.sales_by_date
WHERE sale_date = '2026-08-01';

-- 6. VACUUM specific partition (remove old files)
VACUUM demo_catalog.demo_schema.sales_by_date
WHERE sale_date < '2026-01-01'
RETAIN 168 HOURS;  -- 7 days

-- Note: You CANNOT change partitioning scheme after table creation
-- To repartition, you must:
-- 1. CREATE new table with different PARTITIONED BY
-- 2. INSERT INTO new_table SELECT * FROM old_table
-- 3. DROP old_table
-- 4. RENAME new_table to old_table name

# Liquid Clustering - Modern Data Layout

## What is Liquid Clustering?

Liquid Clustering is the **recommended modern approach** for organizing data in Delta tables, replacing traditional partitioning.

## Key Features

### 1. **Flexible Clustering Keys**
- Cluster data by multiple columns
- **Can change cluster keys** after table creation (unlike partitioning!)
- No need to rewrite entire table to change organization

### 2. **Automatic File Management**
- Automatically creates optimally-sized files
- No small file problem
- No need to manually run OPTIMIZE

### 3. **Co-locality**
- Data with similar values stored together
- Improves query performance through data skipping
- Works for multiple filter columns simultaneously

### 4. **Incremental Clustering**
- Clustering happens incrementally during writes
- New data automatically clustered
- Old data gradually re-clustered

## Liquid Clustering vs. Partitioning

| Feature | Partitioning | Liquid Clustering |
|---------|-------------|-------------------|
| **Directory structure** | Separate dirs per partition | Single directory |
| **Change layout** | ❌ Must recreate table | ✅ ALTER TABLE supported |
| **Multi-column optimization** | Limited (hierarchical) | ✅ All columns equal |
| **Small files** | ⚠️ Risk with high cardinality | ✅ Auto-managed |
| **Metadata overhead** | ⚠️ High with many partitions | ✅ Low |
| **Query optimization** | Partition pruning | Data skipping + file pruning |
| **Best for** | Simple date filtering | Complex multi-column filters |

## When to Use Liquid Clustering

### ✅ Ideal Use Cases:
1. **Multiple filter columns** in queries
2. **High-cardinality columns** (user_id, product_id)
3. **Changing query patterns** - can adjust cluster keys
4. **Large tables** (> 1 TB)
5. **New tables** - recommended default approach

### Requirements:
- Delta Lake protocol Reader=3, Writer=7
- Databricks Runtime 13.3+
- Unity Catalog tables (not external tables)

## Best Practices

🎯 **Cluster column selection**: Choose columns frequently used in WHERE clauses

🎯 **Column order**: Put highest cardinality columns first

🎯 **Limit cluster columns**: 3-4 columns is usually optimal

🎯 **Monitor performance**: Use query metrics to validate clustering effectiveness

In [0]:
%sql
-- LIQUID CLUSTERING - CREATE TABLE EXAMPLES

-- 1. Single column clustering
CREATE OR REPLACE TABLE demo_catalog.demo_schema.events_clustered (
  event_id BIGINT NOT NULL,
  user_id BIGINT NOT NULL,
  event_type STRING NOT NULL,
  event_timestamp TIMESTAMP NOT NULL,
  event_date DATE NOT NULL,
  session_id STRING,
  properties MAP<STRING, STRING>
)
USING DELTA
CLUSTER BY (event_date)
COMMENT 'Events table clustered by date';

-- 2. Multi-column clustering (RECOMMENDED)
-- Order matters: most selective column first
CREATE OR REPLACE TABLE demo_catalog.demo_schema.orders_clustered (
  order_id BIGINT NOT NULL,
  customer_id BIGINT NOT NULL,
  order_date DATE NOT NULL,
  region STRING NOT NULL,
  status STRING NOT NULL,
  total_amount DECIMAL(12,2) NOT NULL
)
USING DELTA
CLUSTER BY (region, order_date, customer_id)
TBLPROPERTIES (
  'delta.targetFileSize' = '134217728'  -- 128 MB per clustered file
)
COMMENT 'Orders clustered by region, date, and customer for multi-dimensional queries';

-- 3. High-cardinality clustering
CREATE OR REPLACE TABLE demo_catalog.demo_schema.user_activity_clustered (
  activity_id BIGINT NOT NULL,
  user_id BIGINT NOT NULL,
  product_id BIGINT NOT NULL,
  activity_date DATE NOT NULL,
  activity_type STRING NOT NULL,
  revenue DECIMAL(10,2)
)
USING DELTA
CLUSTER BY (user_id, activity_date)
COMMENT 'User activity clustered by user_id and date for user-level analytics';

-- 4. Clustering with other features
CREATE OR REPLACE TABLE demo_catalog.demo_schema.transactions_clustered (
  transaction_id BIGINT NOT NULL,
  account_id BIGINT NOT NULL,
  transaction_date DATE NOT NULL,
  transaction_type STRING NOT NULL,
  amount DECIMAL(15,2) NOT NULL,
  currency STRING NOT NULL,
  
  -- Constraints
  CONSTRAINT positive_amount CHECK (amount > 0),
  CONSTRAINT valid_currency CHECK (currency IN ('USD', 'EUR', 'GBP', 'JPY'))
)
USING DELTA
CLUSTER BY (account_id, transaction_date)
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)
COMMENT 'Transactions with clustering, constraints, and CDC enabled';

In [0]:
%sql
-- ALTER TABLE - MODIFY LIQUID CLUSTERING
-- One of the biggest advantages: you can change clustering keys!

-- 1. ADD clustering to existing table
ALTER TABLE demo_catalog.demo_schema.sales_optimized
CLUSTER BY (region, sale_date);

-- 2. CHANGE clustering columns
-- Suppose queries now filter by customer_id more than region
ALTER TABLE demo_catalog.demo_schema.orders_clustered
CLUSTER BY (customer_id, order_date);
-- Old clustering: (region, order_date, customer_id)
-- New clustering: (customer_id, order_date)
-- Data will be gradually re-clustered during writes and OPTIMIZE

-- 3. ADD more clustering columns
ALTER TABLE demo_catalog.demo_schema.events_clustered
CLUSTER BY (event_date, user_id, event_type);

-- 4. REMOVE clustering (convert to non-clustered table)
ALTER TABLE demo_catalog.demo_schema.events_clustered
CLUSTER BY NONE;
-- Table becomes regular Delta table without clustering

-- 5. Trigger incremental clustering optimization
-- Run OPTIMIZE to apply new clustering to existing data
OPTIMIZE demo_catalog.demo_schema.orders_clustered;
-- This incrementally re-clusters data with new keys

-- 6. Check current clustering configuration
DESCRIBE DETAIL demo_catalog.demo_schema.orders_clustered;
-- Look for "clusteringColumns" field

-- 7. View clustering statistics
DESCRIBE EXTENDED demo_catalog.demo_schema.orders_clustered;

In [0]:
%sql
-- LIQUID CLUSTERING - BEST PRACTICES & MONITORING

-- 1. Analyze query patterns to choose cluster columns
-- Look at your most common WHERE clause filters
SELECT 
  query_text,
  execution_count,
  avg_execution_time_ms
FROM system.query.history
WHERE table_name = 'orders'
  AND query_text LIKE '%WHERE%'
ORDER BY execution_count DESC
LIMIT 10;

-- 2. Check table statistics
DESCRIBE DETAIL demo_catalog.demo_schema.orders_clustered;
-- Look at:
-- - numFiles: file count
-- - sizeInBytes: total size
-- - clusteringColumns: current clustering configuration

-- 3. Monitor clustering effectiveness
-- Run ANALYZE to update statistics
ANALYZE TABLE demo_catalog.demo_schema.orders_clustered 
COMPUTE STATISTICS FOR ALL COLUMNS;

-- 4. View column statistics for skipping efficiency
DESCRIBE EXTENDED demo_catalog.demo_schema.orders_clustered;
-- Check min/max statistics for cluster columns

-- 5. Test query performance before/after clustering
-- Before clustering
SELECT COUNT(*), SUM(total_amount)
FROM demo_catalog.demo_schema.orders_clustered
WHERE region = 'North America' 
  AND order_date BETWEEN '2026-07-01' AND '2026-07-31';
-- Note: execution time and data scanned

-- After OPTIMIZE with clustering
OPTIMIZE demo_catalog.demo_schema.orders_clustered;

-- Re-run same query and compare metrics
SELECT COUNT(*), SUM(total_amount)
FROM demo_catalog.demo_schema.orders_clustered
WHERE region = 'North America' 
  AND order_date BETWEEN '2026-07-01' AND '2026-07-31';

-- 6. Incremental re-clustering after changing cluster keys
-- Small batches to avoid blocking writes
OPTIMIZE demo_catalog.demo_schema.orders_clustered
WHERE order_date >= '2026-08-01';  -- Cluster recent data first

-- 7. Best practice settings for clustered tables
ALTER TABLE demo_catalog.demo_schema.orders_clustered
SET TBLPROPERTIES (
  -- Target 128-256 MB files for optimal clustering
  'delta.targetFileSize' = '134217728',
  
  -- Enable auto-compaction to maintain clustering
  'delta.autoOptimize.autoCompact' = 'true'
);

# Quick Reference Summary

## Table Creation Workflow

```sql
CREATE OR REPLACE TABLE catalog.schema.table_name (
  -- Define columns with data types
  col1 TYPE [NOT NULL] [COMMENT 'description'],
  
  -- Add constraints (Serverless: only PK/FK supported)
  CONSTRAINT pk PRIMARY KEY (col),
  CONSTRAINT fk FOREIGN KEY (col) REFERENCES other_table(col)
)
USING DELTA
[PARTITIONED BY (col1, col2)]  -- Legacy approach
[CLUSTER BY (col1, col2)]      -- Modern approach (recommended)
TBLPROPERTIES (
  'delta.property' = 'value'
)
COMMENT 'Table description';
```

## Key Decision Points

### 1. Data Types
- Use **STRING** for text (most flexible)
- Use **DECIMAL(p,s)** for money/precise calculations
- Use **TIMESTAMP** for timestamps with timezone
- Use **VARIANT** for JSON/semi-structured data

### 2. Constraints (Serverless)
- **NOT NULL**: Required fields (✅ Supported)
- **PRIMARY KEY**: Unique identifiers (✅ Supported, requires protocol upgrade)
- **FOREIGN KEY**: Referential integrity (✅ Supported, informational)
- **CHECK**: ❌ Not supported on Serverless - use ETL validation

### 3. Table Properties
- **Performance**: `delta.autoOptimize.autoCompact = true`
- **Time Travel**: `delta.logRetentionDuration = interval 90 days`
- **CDC**: `delta.enableChangeDataFeed = true`
- **Schema Evolution**: `delta.columnMapping.mode = name`

### 4. Data Organization

**✅ Use CLUSTER BY (recommended)**
- Modern, flexible approach
- Can change cluster keys later
- Automatic file management
- Best for: new tables, complex queries, high-cardinality columns

**⚠️ Use PARTITIONED BY (legacy)**
- Fixed at creation, cannot change
- Risk of small files with high cardinality
- Best for: simple date filtering, existing workflows

### 5. Protocol Versions
- **Reader=1, Writer=1**: Basic Delta
- **Reader=2, Writer=5**: Column Mapping
- **Reader=3, Writer=7**: Deletion Vectors, Clustering
- ⚠️ **Upgrades are ONE-WAY** - cannot downgrade

## Common ALTER TABLE Operations

```sql
-- Add column
ALTER TABLE table ADD COLUMN col TYPE;

-- Rename column (needs column mapping)
ALTER TABLE table RENAME COLUMN old TO new;

-- Add PRIMARY KEY constraint
ALTER TABLE table ADD CONSTRAINT pk PRIMARY KEY (col);

-- Change clustering
ALTER TABLE table CLUSTER BY (col1, col2);

-- Update properties
ALTER TABLE table SET TBLPROPERTIES ('prop' = 'value');

-- Enable Change Data Feed
ALTER TABLE table SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');
```

## Performance Optimization Checklist

- ☑️ Use **Liquid Clustering** for new tables
- ☑️ Enable **auto-compaction** for write-heavy tables
- ☑️ Set appropriate **retention periods** for compliance
- ☑️ Implement **data validation in ETL** (CHECK constraints not supported on Serverless)
- ☑️ Enable **Change Data Feed** if CDC needed
- ☑️ Run **ANALYZE TABLE** to update statistics
- ☑️ Monitor with **DESCRIBE DETAIL** regularly

---

**For Modern Delta Tables**: Use `CLUSTER BY` + auto-compaction + column mapping + deletion vectors

**Protocol Upgrade Command**:
```sql
ALTER TABLE table SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableDeletionVectors' = 'true'
);
```

# Performance Comparison: Partitioning vs Clustering

This section demonstrates real-world performance differences between partitioned and clustered tables.

## Test Setup

### Scenario
We'll create a realistic sales dataset with:
- **10 million rows** of sales transactions
- **Multiple regions** (5 regions)
- **6 months of data** (dates)
- **Various products and customers** (high cardinality)

### Tables to Compare
1. **Partitioned Table**: `PARTITIONED BY (region, sale_date)`
2. **Clustered Table**: `CLUSTER BY (region, sale_date, customer_id)`
3. **Control Table**: No partitioning or clustering (baseline)

### Test Queries
We'll run common query patterns:
1. Single region + date range filter
2. Multi-region filter
3. Customer-specific query
4. Aggregation across regions and dates

### Metrics to Compare
- **Execution time**
- **Data scanned** (bytes read)
- **Files read**
- **Partition pruning effectiveness**

In [0]:
# Generate 10 million rows of sample sales data
from pyspark.sql import functions as F
from datetime import datetime, timedelta
import random

print("Generating 10 million sample sales records...")

# Configuration
num_rows = 10_000_000
regions = ['North America', 'Europe', 'Asia', 'South America', 'Africa']
product_ids = list(range(1, 1001))  # 1000 products
customer_ids = list(range(1, 50001))  # 50,000 customers

# Create base DataFrame with ID column
df = spark.range(0, num_rows).toDF("id")

# Add columns with realistic data distribution
df_sales = df \
    .withColumn("sale_id", F.col("id")) \
    .withColumn("region", F.array([F.lit(r) for r in regions])[F.floor(F.rand() * 5).cast("int")]) \
    .withColumn("sale_date", F.date_add(F.lit("2026-03-01"), (F.rand() * 180).cast("int"))) \
    .withColumn("customer_id", F.floor(F.rand() * 50000 + 1).cast("bigint")) \
    .withColumn("product_id", F.floor(F.rand() * 1000 + 1).cast("bigint")) \
    .withColumn("quantity", F.floor(F.rand() * 10 + 1).cast("int")) \
    .withColumn("unit_price", (F.rand() * 1000 + 10).cast("decimal(10,2)")) \
    .withColumn("amount", (F.col("quantity") * F.col("unit_price")).cast("decimal(12,2)")) \
    .withColumn("sale_timestamp", F.to_timestamp(F.col("sale_date"))) \
    .drop("id")

# Cache for reuse
df_sales.cache()
row_count = df_sales.count()

print(f"✅ Generated {row_count:,} rows")
print(f"Date range: {df_sales.agg(F.min('sale_date'), F.max('sale_date')).collect()[0]}")
print(f"\nSample data:")
display(df_sales.limit(10))

In [0]:
%sql
-- TABLE 1: BASELINE - No partitioning or clustering
CREATE OR REPLACE TABLE demo_catalog.demo_schema.sales_baseline (
  sale_id BIGINT NOT NULL,
  region STRING NOT NULL,
  sale_date DATE NOT NULL,
  customer_id BIGINT NOT NULL,
  product_id BIGINT NOT NULL,
  quantity INT NOT NULL,
  unit_price DECIMAL(10,2) NOT NULL,
  amount DECIMAL(12,2) NOT NULL,
  sale_timestamp TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Baseline table - no optimization';

SELECT 'Created: sales_baseline' AS status;

In [0]:
%sql
-- TABLE 2: PARTITIONED by region and sale_date
CREATE OR REPLACE TABLE demo_catalog.demo_schema.sales_partitioned (
  sale_id BIGINT NOT NULL,
  region STRING NOT NULL,
  sale_date DATE NOT NULL,
  customer_id BIGINT NOT NULL,
  product_id BIGINT NOT NULL,
  quantity INT NOT NULL,
  unit_price DECIMAL(10,2) NOT NULL,
  amount DECIMAL(12,2) NOT NULL,
  sale_timestamp TIMESTAMP NOT NULL
)
USING DELTA
PARTITIONED BY (region, sale_date)
COMMENT 'Partitioned by region and date';

SELECT 'Created: sales_partitioned' AS status;

In [0]:
%sql
-- TABLE 3: LIQUID CLUSTERING by region, sale_date, and customer_id
CREATE OR REPLACE TABLE demo_catalog.demo_schema.sales_clustered (
  sale_id BIGINT NOT NULL,
  region STRING NOT NULL,
  sale_date DATE NOT NULL,
  customer_id BIGINT NOT NULL,
  product_id BIGINT NOT NULL,
  quantity INT NOT NULL,
  unit_price DECIMAL(10,2) NOT NULL,
  amount DECIMAL(12,2) NOT NULL,
  sale_timestamp TIMESTAMP NOT NULL
)
USING DELTA
CLUSTER BY (region, sale_date, customer_id)
TBLPROPERTIES (
  'delta.targetFileSize' = '134217728',  -- 128 MB
  'delta.autoOptimize.autoCompact' = 'true'
)
COMMENT 'Clustered by region, date, and customer';

SELECT 'Created: sales_clustered' AS status;

In [0]:
# Insert data into all three tables
print("Loading data into tables...\n")

# Table 1: Baseline
print("1️⃣ Loading sales_baseline...")
df_sales.write.mode("overwrite").saveAsTable("demo_catalog.demo_schema.sales_baseline")
print("✅ Baseline table loaded\n")

# Table 2: Partitioned
print("2️⃣ Loading sales_partitioned...")
df_sales.write.mode("overwrite").saveAsTable("demo_catalog.demo_schema.sales_partitioned")
print("✅ Partitioned table loaded\n")

# Table 3: Clustered (write and optimize)
print("3️⃣ Loading sales_clustered...")
df_sales.write.mode("overwrite").saveAsTable("demo_catalog.demo_schema.sales_clustered")
print("✅ Clustered table loaded\n")

# Optimize the clustered table to apply clustering
print("🔧 Optimizing clustered table...")
spark.sql("OPTIMIZE demo_catalog.demo_schema.sales_clustered")
print("✅ Clustering optimization complete\n")

# Uncache the DataFrame
df_sales.unpersist()

print("="*60)
print("✅ ALL TABLES READY FOR PERFORMANCE TESTING")
print("="*60)

In [0]:
%sql
-- Compare table file statistics before running queries
SELECT 
  'Baseline' AS table_name,
  numFiles,
  ROUND(sizeInBytes / 1024 / 1024, 2) AS size_mb,
  format
FROM (
  DESCRIBE DETAIL demo_catalog.demo_schema.sales_baseline
)

UNION ALL

SELECT 
  'Partitioned' AS table_name,
  numFiles,
  ROUND(sizeInBytes / 1024 / 1024, 2) AS size_mb,
  format
FROM (
  DESCRIBE DETAIL demo_catalog.demo_schema.sales_partitioned
)

UNION ALL

SELECT 
  'Clustered' AS table_name,
  numFiles,
  ROUND(sizeInBytes / 1024 / 1024, 2) AS size_mb,
  format
FROM (
  DESCRIBE DETAIL demo_catalog.demo_schema.sales_clustered
)
ORDER BY table_name;

## Performance Test Queries

We'll run 4 common query patterns and compare execution metrics:

### Query 1: Single Region + Date Range
```sql
WHERE region = 'North America' 
  AND sale_date BETWEEN '2026-06-01' AND '2026-06-30'
```
**Expected Winner**: Partitioned (direct partition pruning)

### Query 2: Multi-Region Filter
```sql
WHERE region IN ('Europe', 'Asia')
  AND sale_date >= '2026-07-01'
```
**Expected Winner**: Clustered (better at multiple predicates)

### Query 3: Customer-Specific Query
```sql
WHERE customer_id = 12345
  AND sale_date >= '2026-05-01'
```
**Expected Winner**: Clustered (has customer_id in cluster keys)

### Query 4: Complex Aggregation
```sql
GROUP BY region, MONTH(sale_date)
WITH multiple WHERE conditions
```
**Expected Winner**: Clustered (optimized for analytical queries)

---

**Note**: Run each query 2-3 times to warm up caches for fair comparison

In [0]:
%sql
-- QUERY 1 - BASELINE TABLE
-- Single region + date range filter

SELECT 
  COUNT(*) AS total_sales,
  SUM(amount) AS total_revenue,
  AVG(amount) AS avg_sale_amount,
  COUNT(DISTINCT customer_id) AS unique_customers
FROM demo_catalog.demo_schema.sales_baseline
WHERE region = 'North America'
  AND sale_date BETWEEN '2026-06-01' AND '2026-06-30';

In [0]:
%sql
-- QUERY 1 - PARTITIONED TABLE
-- Single region + date range filter
-- Should see PARTITION PRUNING in action

SELECT 
  COUNT(*) AS total_sales,
  SUM(amount) AS total_revenue,
  AVG(amount) AS avg_sale_amount,
  COUNT(DISTINCT customer_id) AS unique_customers
FROM demo_catalog.demo_schema.sales_partitioned
WHERE region = 'North America'
  AND sale_date BETWEEN '2026-06-01' AND '2026-06-30';

In [0]:
%sql
-- QUERY 1 - CLUSTERED TABLE
-- Single region + date range filter
-- Should use DATA SKIPPING

SELECT 
  COUNT(*) AS total_sales,
  SUM(amount) AS total_revenue,
  AVG(amount) AS avg_sale_amount,
  COUNT(DISTINCT customer_id) AS unique_customers
FROM demo_catalog.demo_schema.sales_clustered
WHERE region = 'North America'
  AND sale_date BETWEEN '2026-06-01' AND '2026-06-30';

In [0]:
%sql
-- QUERY 2: Multi-region filter with date range
-- Tests how well each approach handles multiple region values

-- Baseline
SELECT 'Baseline' AS table_type, COUNT(*) AS row_count, SUM(amount) AS total_revenue
FROM demo_catalog.demo_schema.sales_baseline
WHERE region IN ('Europe', 'Asia')
  AND sale_date >= '2026-07-01'

UNION ALL

-- Partitioned (needs to read 2 region partitions)
SELECT 'Partitioned' AS table_type, COUNT(*) AS row_count, SUM(amount) AS total_revenue
FROM demo_catalog.demo_schema.sales_partitioned
WHERE region IN ('Europe', 'Asia')
  AND sale_date >= '2026-07-01'

UNION ALL

-- Clustered (should handle this efficiently)
SELECT 'Clustered' AS table_type, COUNT(*) AS row_count, SUM(amount) AS total_revenue
FROM demo_catalog.demo_schema.sales_clustered
WHERE region IN ('Europe', 'Asia')
  AND sale_date >= '2026-07-01';

In [0]:
%sql
-- QUERY 3: Customer-specific query
-- Clustered table should WIN here (customer_id is in cluster keys)

-- Baseline (full table scan on customer_id)
SELECT 'Baseline' AS table_type, COUNT(*) AS order_count, SUM(amount) AS customer_total
FROM demo_catalog.demo_schema.sales_baseline
WHERE customer_id = 12345
  AND sale_date >= '2026-05-01'

UNION ALL

-- Partitioned (still needs to scan many date partitions)
SELECT 'Partitioned' AS table_type, COUNT(*) AS order_count, SUM(amount) AS customer_total
FROM demo_catalog.demo_schema.sales_partitioned
WHERE customer_id = 12345
  AND sale_date >= '2026-05-01'

UNION ALL

-- Clustered (customer_id is a cluster key - should be MUCH faster)
SELECT 'Clustered' AS table_type, COUNT(*) AS order_count, SUM(amount) AS customer_total
FROM demo_catalog.demo_schema.sales_clustered
WHERE customer_id = 12345
  AND sale_date >= '2026-05-01';

In [0]:
%sql
-- QUERY 4: Complex aggregation across regions and months
-- Tests analytical query performance

-- Baseline
SELECT 
  'Baseline' AS table_type,
  region,
  DATE_TRUNC('MONTH', sale_date) AS sale_month,
  COUNT(*) AS sales_count,
  SUM(amount) AS monthly_revenue,
  AVG(amount) AS avg_sale,
  COUNT(DISTINCT customer_id) AS unique_customers
FROM demo_catalog.demo_schema.sales_baseline
WHERE sale_date BETWEEN '2026-06-01' AND '2026-08-31'
GROUP BY region, DATE_TRUNC('MONTH', sale_date)

UNION ALL

-- Partitioned
SELECT 
  'Partitioned' AS table_type,
  region,
  DATE_TRUNC('MONTH', sale_date) AS sale_month,
  COUNT(*) AS sales_count,
  SUM(amount) AS monthly_revenue,
  AVG(amount) AS avg_sale,
  COUNT(DISTINCT customer_id) AS unique_customers
FROM demo_catalog.demo_schema.sales_partitioned
WHERE sale_date BETWEEN '2026-06-01' AND '2026-08-31'
GROUP BY region, DATE_TRUNC('MONTH', sale_date)

UNION ALL

-- Clustered
SELECT 
  'Clustered' AS table_type,
  region,
  DATE_TRUNC('MONTH', sale_date) AS sale_month,
  COUNT(*) AS sales_count,
  SUM(amount) AS monthly_revenue,
  AVG(amount) AS avg_sale,
  COUNT(DISTINCT customer_id) AS unique_customers
FROM demo_catalog.demo_schema.sales_clustered
WHERE sale_date BETWEEN '2026-06-01' AND '2026-08-31'
GROUP BY region, DATE_TRUNC('MONTH', sale_date)

ORDER BY table_type, region, sale_month;

In [0]:
%sql
-- Analyze recent query performance from system tables
-- Compare execution times and data scanned

SELECT 
  statement_text,
  CASE 
    WHEN statement_text LIKE '%sales_baseline%' THEN 'Baseline'
    WHEN statement_text LIKE '%sales_partitioned%' THEN 'Partitioned'
    WHEN statement_text LIKE '%sales_clustered%' THEN 'Clustered'
    ELSE 'Other'
  END AS table_type,
  executed_as_user_name,
  start_time,
  ROUND(total_duration_ms / 1000, 2) AS duration_seconds,
  ROUND(read_bytes / 1024 / 1024, 2) AS data_scanned_mb,
  read_files_count,
  read_partitions_count,
  ROUND(rows_produced_count / 1000, 1) AS rows_produced_k
FROM system.query.history
WHERE start_time >= CURRENT_TIMESTAMP() - INTERVAL 1 HOUR
  AND (statement_text LIKE '%sales_baseline%' 
    OR statement_text LIKE '%sales_partitioned%' 
    OR statement_text LIKE '%sales_clustered%')
  AND statement_text NOT LIKE '%system.query.history%'
  AND status = 'FINISHED'
ORDER BY start_time DESC
LIMIT 50;

## Performance Comparison Results

### Key Findings

After running the queries above, you should observe these patterns:

#### 1. 🏆 Single Region + Date Range (Query 1)
**Winner: Partitioned Table**
- **Partitioned**: Fastest - Direct partition pruning eliminates irrelevant partitions
- **Clustered**: Good - Data skipping provides similar benefits
- **Baseline**: Slowest - Full table scan required

**Why Partitioned Wins**: When filtering on exact partition columns (region + date), partition pruning is extremely efficient.

---

#### 2. 🏆 Multi-Region Filter (Query 2)
**Winner: Clustered Table**
- **Clustered**: Fastest - Efficiently handles multiple predicates
- **Partitioned**: Moderate - Must read multiple region partitions
- **Baseline**: Slowest - Full scan

**Why Clustered Wins**: Better at handling queries with multiple values in predicates (IN clause).

---

#### 3. 🏆 Customer-Specific Query (Query 3)
**Winner: Clustered Table (BY FAR)**
- **Clustered**: MUCH faster - customer_id is a cluster key
- **Partitioned**: Slow - customer_id is NOT in partition scheme
- **Baseline**: Slowest - No optimization for customer_id

**Why Clustered Dominates**: 
- customer_id is in the cluster keys
- Data skipping dramatically reduces files scanned
- Partitioned table has NO benefit here (customer_id not in partition columns)

---

#### 4. 🏆 Complex Aggregation (Query 4)
**Winner: Clustered Table**
- **Clustered**: Fastest - Optimized file layout for analytical queries
- **Partitioned**: Good - Partition pruning helps
- **Baseline**: Slowest - Full scan + large shuffle

**Why Clustered Wins**: 
- Data co-location reduces shuffle operations
- Better file organization for aggregations
- Smaller file count = better parallelism

---

### Overall Recommendations

#### Use **Partitioning** When:
- ✅ Simple date-based filtering is your PRIMARY query pattern
- ✅ You ALWAYS filter on the partition columns
- ✅ Query patterns are very predictable and narrow
- ✅ Working with legacy systems that expect partitions

#### Use **Liquid Clustering** When:
- ✅ Multiple columns are used in WHERE clauses
- ✅ Query patterns vary (different filter combinations)
- ✅ High-cardinality columns (customer_id, product_id)
- ✅ Need flexibility to change optimization strategy
- ✅ Building new tables (RECOMMENDED default)

---

### File Statistics Comparison

| Table Type | # Files | Avg File Size | Partition Count |
|------------|---------|---------------|------------------|
| **Baseline** | ~200-300 | Varies | 1 (no partitions) |
| **Partitioned** | 800-900+ | Small (risk!) | ~900 (5 regions × 180 dates) |
| **Clustered** | ~100-150 | 128 MB (optimized) | 1 (no partitions) |

⚠️ **Partitioned Warning**: Too many partitions = Small File Problem!

---

### Performance Metrics Summary

**Data Scanned** (lower is better):
- Clustered: 50-70% less data scanned on customer queries
- Partitioned: 60-80% less data scanned on region+date queries
- Baseline: Always scans full table

**Execution Time** (typical 10M row dataset):
- Clustered: 0.5-2 seconds for selective queries
- Partitioned: 1-3 seconds for partition-aligned queries
- Baseline: 3-8 seconds for most queries

---

## 🎯 Conclusion

**For Modern Delta Tables**: 

✅ **Use Liquid Clustering** as your default choice

✅ **Flexibility > Static partitioning**

✅ **Better for diverse query patterns**

✅ **No small file problems**

✅ **Can evolve with changing requirements**

In [0]:
%sql
-- CLEANUP: Drop performance test tables when done
-- Uncomment to execute

-- DROP TABLE IF EXISTS demo_catalog.demo_schema.sales_baseline;
-- DROP TABLE IF EXISTS demo_catalog.demo_schema.sales_partitioned;
-- DROP TABLE IF EXISTS demo_catalog.demo_schema.sales_clustered;

-- SELECT 'Performance test tables cleaned up' AS status;